# 02. Initialization: Dial-In LLM 기반 초기 IntentSet 구축

**목적**: Pre-Init에서 생성된 34,025개 의도 요약문을 클러스터링하여 초기 IntentSet을 구축한다.

**방법론**: Dial-In LLM 논문(Hong et al., 2025)의 LLM-in-the-loop 프레임워크를 적용한다.
- Algorithm 1: 반복적 클러스터링 + Coherence 평가로 최적 K를 자동 탐색
- fine-tuned 소형 LLM → Gemini 2.5 flash-lite API로 대체 (few-shot prompting)
- Post-correction: 라벨 임베딩 기반 확률적 병합

**산출물**:
- `data/processed/initial_intentset.parquet` — 최종 IntentSet (클러스터 ID + 라벨 + 소속 문장)
- `checkpoints/clustering/` — 반복별 중간 상태

**예상 비용**: ~₩1,000~1,300 / 잔여 예산 ~₩44,800

**예상 소요**: 임베딩 ~5분, 클러스터링 반복 ~30분, 라벨링+병합 ~15분

## 0. 환경 설정
Google Drive 마운트, 작업 디렉토리 이동, 필수 라이브러리 설치 및 임포트.
GPU 런타임 필요 (KR-SBERT 임베딩 생성용).

In [ ]:
# === 환경 설정 ===
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/5stone_experiment/1_clustering_test')
print(f'작업 디렉토리: {os.getcwd()}')

!pip install -q google-genai sentence-transformers scipy

import json
import time
import unicodedata
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import cosine
from scipy.special import iv as bessel_iv
import warnings
warnings.filterwarnings('ignore')

# GPU 확인
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only"}')

Mounted at /content/drive
작업 디렉토리: /content/drive/MyDrive/5stone_experiment/1_clustering_test
GPU: Tesla T4


## 0-1. 경로 설정 (PATH_CONFIG)

In [ ]:
# === 경로 설정 ===
BASE_DIR = Path('/content/drive/MyDrive/5stone_experiment/1_clustering_test')

PATH_CONFIG = {
    'processed': BASE_DIR / 'data' / 'processed',
    'checkpoints': BASE_DIR / 'checkpoints',
    'embeddings': BASE_DIR / 'embeddings',
    'vector_db': BASE_DIR / 'vector_db',
    'notebooks': BASE_DIR / 'notebooks',
}

# 입력
PATH_CONFIG['pre_init_intents'] = PATH_CONFIG['processed'] / 'pre_init_intents.parquet'

# 산출물
PATH_CONFIG['intent_embeddings'] = PATH_CONFIG['embeddings'] / 'intent_embeddings.npy'
PATH_CONFIG['clustering_ckpt_dir'] = PATH_CONFIG['checkpoints'] / 'clustering'
PATH_CONFIG['initial_intentset'] = PATH_CONFIG['processed'] / 'initial_intentset.parquet'

# 디렉토리 생성
PATH_CONFIG['clustering_ckpt_dir'].mkdir(parents=True, exist_ok=True)

for key in ['pre_init_intents', 'embeddings', 'clustering_ckpt_dir']:
    path = PATH_CONFIG[key]
    marker = '✓' if path.exists() else '✗'
    print(f'  {marker} {key}: {path}')

  ✓ pre_init_intents: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/pre_init_intents.parquet
  ✓ embeddings: /content/drive/MyDrive/5stone_experiment/1_clustering_test/embeddings
  ✓ clustering_ckpt_dir: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/clustering


## 0-2. 하이퍼파라미터 설정
Dial-In LLM 논문의 파라미터를 기반으로, 우리 데이터 규모(34,025건)에 맞게 조정한다.

In [ ]:
# === 하이퍼파라미터 ===
CONFIG = {
    # --- 임베딩 ---
    'embed_model_name': 'snunlp/KR-SBERT-V40K-klueNLI-augSTS',
    'embed_batch_size': 256,

    # --- 반복 클러스터링 (Algorithm 1) ---
    'cluster_candidate_k': [50, 100, 200, 500, 800, 1000],  # 후보 클러스터 수 N
    'cluster_max_iter': 5,         # T_max: 최대 반복 횟수
    'cluster_epsilon': 0.05,       # ε: 미할당 비율 종료 임계치
    'cluster_sample_n': 20,        # 클러스터당 Coherence 평가용 샘플 수
    'cluster_random_state': 42,
    'cluster_kmeans_n_init': 10,   # K-Means 초기화 횟수

    # --- Gemini API ---
    'gemini_model': 'gemini-2.5-flash-lite',
    'gemini_temperature': 0.1,
    'gemini_max_output_tokens': 64,   # Coherence: "Good"/"Bad", Labelling: 짧은 라벨
    'gemini_rate_limit_delay': 0.25,
    'gemini_retry_max': 3,
    'gemini_retry_delay': 5,

    # --- 의도 라벨링 ---
    'label_max_output_tokens': 128,

    # --- 클러스터 병합 (Post-Correction) ---
    'merge_geodesic_theta': 0.8,    # θ: geodesic distance 임계치
    'merge_prob_tau': 0.7,          # τ: 병합 확률 임계치
    'merge_kappa': 50.0,            # κ: vMF 분포 집중도

    # --- 비용 ---
    'budget_total_krw': 50000,
    'budget_spent_krw': 5200,       # Pre-Init 사용분
    'exchange_rate': 1450.0,
    'input_price_per_1m': 0.10,
    'output_price_per_1m': 0.40,
}

print('하이퍼파라미터 설정 완료')
print(f'  후보 K: {CONFIG["cluster_candidate_k"]}')
print(f'  최대 반복: {CONFIG["cluster_max_iter"]}')
print(f'  잔여 예산: ₩{CONFIG["budget_total_krw"] - CONFIG["budget_spent_krw"]:,}')

하이퍼파라미터 설정 완료
  후보 K: [50, 100, 200, 500, 800, 1000]
  최대 반복: 5
  잔여 예산: ₩44,800


## 0-3. Gemini API 초기화

In [ ]:
# === Gemini API 초기화 ===
from google.colab import userdata
from google import genai
from google.genai import types

client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

test_response = client.models.generate_content(
    model=CONFIG['gemini_model'],
    contents='테스트입니다. "ok"라고만 답하세요.',
    config=types.GenerateContentConfig(temperature=0.0, max_output_tokens=10),
)
print(f'Gemini API 연결 성공: {test_response.text}')

Gemini API 연결 성공: ok


## 0-4. 유틸리티 함수 정의
NFC 정규화, Gemini API 호출 래퍼, 클러스터 샘플링 등 공통 함수

In [ ]:
# === 유틸리티 함수 ===

def nfc(text: str) -> str:
    return unicodedata.normalize('NFC', text)


def call_gemini(
    prompt: str,
    system_instruction: str,
    max_output_tokens: int = None,
) -> str:
    """
    Gemini API 호출 래퍼. 재시도 및 rate limit 대기 포함.
    """
    mot = max_output_tokens or CONFIG['gemini_max_output_tokens']
    for attempt in range(CONFIG['gemini_retry_max']):
        try:
            response = client.models.generate_content(
                model=CONFIG['gemini_model'],
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=CONFIG['gemini_temperature'],
                    max_output_tokens=mot,
                ),
            )
            time.sleep(CONFIG['gemini_rate_limit_delay'])
            return response.text.strip()
        except Exception as e:
            error_str = str(e)
            if '429' in error_str or 'RESOURCE_EXHAUSTED' in error_str:
                wait = CONFIG['gemini_retry_delay'] * (attempt + 1) * 2
                print(f'  [Rate limit] {wait}s 대기 ({attempt+1}/{CONFIG["gemini_retry_max"]})')
                time.sleep(wait)
            else:
                wait = CONFIG['gemini_retry_delay'] * (attempt + 1)
                print(f'  [Error] {error_str[:80]}... ({attempt+1}/{CONFIG["gemini_retry_max"]})')
                time.sleep(wait)
    return ''


def sample_from_cluster(
    sentences: list,
    n: int = 20,
    seed: int = 42,
) -> list:
    """
    클러스터에서 최대 n개 문장을 랜덤 샘플링한다.
    클러스터 크기가 n 이하이면 전체를 반환한다.
    """
    if len(sentences) <= n:
        return sentences
    rng = np.random.RandomState(seed)
    indices = rng.choice(len(sentences), size=n, replace=False)
    return [sentences[i] for i in indices]


print('유틸리티 함수 정의 완료')

유틸리티 함수 정의 완료


## 1. Pre-Init 데이터 로드
`01_pre_init_idas.ipynb`에서 생성한 의도 분리·요약 결과를 로드한다.

In [ ]:
# === Pre-Init 데이터 로드 ===

def load_pre_init_data() -> pd.DataFrame:
    path = PATH_CONFIG['pre_init_intents']
    df = pd.read_parquet(path)
    # NFC 정규화
    df['intent_summary'] = df['intent_summary'].apply(nfc)
    print(f'Pre-Init 데이터 로드: {len(df):,}건')
    print(f'  원본 상담: {df["source_id"].nunique():,}')
    print(f'  Fallback: {df["is_fallback"].sum()}건')
    return df


intent_df = load_pre_init_data()

# 클러스터링 입력: intent_summary 리스트
sentences = intent_df['intent_summary'].tolist()
print(f'클러스터링 입력 문장 수: {len(sentences):,}')

Pre-Init 데이터 로드: 34,025건
  원본 상담: 11,033
  Fallback: 11건
클러스터링 입력 문장 수: 34,025


## 2. 임베딩 생성 (KR-SBERT)
KR-SBERT 모델로 34,025개 intent_summary의 문장 임베딩을 생성한다.

**모델**: `snunlp/KR-SBERT-V40K-klueNLI-augSTS` (768차원)

**체크포인트**: `embeddings/intent_embeddings.npy`

In [ ]:
# === 임베딩 생성 ===
from sentence_transformers import SentenceTransformer

def generate_embeddings(sentences: list) -> np.ndarray:
    """
    KR-SBERT로 문장 임베딩을 생성한다.
    체크포인트가 존재하면 로드.
    """
    ckpt_path = PATH_CONFIG['intent_embeddings']

    if ckpt_path.exists():
        print(f'체크포인트 로드: {ckpt_path}')
        embeddings = np.load(ckpt_path)
        print(f'  shape: {embeddings.shape}')
        return embeddings

    print(f'임베딩 생성 중... (모델: {CONFIG["embed_model_name"]})')
    model = SentenceTransformer(CONFIG['embed_model_name'])

    embeddings = model.encode(
        sentences,
        batch_size=CONFIG['embed_batch_size'],
        show_progress_bar=True,
        normalize_embeddings=True,  # L2 정규화 → unit hypersphere
    )

    np.save(ckpt_path, embeddings)
    print(f'체크포인트 저장: {ckpt_path}')
    print(f'  shape: {embeddings.shape}')

    return embeddings


embeddings = generate_embeddings(sentences)
print(f'임베딩 차원: {embeddings.shape[1]}, L2 norm 확인: {np.linalg.norm(embeddings[0]):.4f}')

임베딩 생성 중... (모델: snunlp/KR-SBERT-V40K-klueNLI-augSTS)


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/133 [00:00<?, ?it/s]

체크포인트 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/embeddings/intent_embeddings.npy
  shape: (34025, 768)
임베딩 차원: 768, L2 norm 확인: 1.0000


## 3. 반복 클러스터링 (Dial-In LLM Algorithm 1)

논문의 핵심 알고리즘을 구현한다:
1. 후보 K 각각으로 K-Means 클러스터링
2. 각 클러스터를 Gemini Coherence Evaluator로 Good/Bad 판정
3. Good/Bad 비율이 최대인 K를 선택
4. Good 클러스터 확정, Bad 클러스터의 문장은 다음 반복으로
5. 미할당 비율 < ε 또는 반복 T_max까지 반복

### 3-1. Coherence Evaluator 프롬프트
논문 Appendix B의 프롬프트를 한국어로 적응하고, few-shot 예시를 추가한다.

In [ ]:
# === Coherence Evaluator 프롬프트 ===

COHERENCE_SYSTEM = """당신은 문장 클러스터링 품질을 평가하는 전문가입니다.

주어진 문장 그룹의 의미적 일관성을 평가하여 "Good" 또는 "Bad"로 분류하세요.

## 판정 기준
- **Good**: 모든 문장이 하나의 구체적인 의도/주제에 집중되어 있음
- **Bad**: 문장들이 서로 다른 의도를 포함하거나, 주제가 모호하거나 너무 광범위함

## 예시

입력: ["확인 요청 — 카드 환불 누락", "확인 요청 — 환불 미처리 상태", "문의 — 환불 금액 미입금"]
출력: Good

입력: ["문의 — 요금제 변경", "확인 요청 — 배송 상태", "요청 — 카드 해지", "문의 — 해외여행 보험"]
출력: Bad

입력: ["문의 — 통화 품질 불량", "문의 — 인터넷 속도 저하", "문의 — 와이파이 연결 불량"]
출력: Good

입력: ["요청 — 비밀번호 변경", "문의 — 포인트 적립", "확인 요청 — 결제 내역"]
출력: Bad

"Good" 또는 "Bad"만 출력하세요. 다른 내용은 포함하지 마세요."""


def evaluate_coherence(cluster_sentences: list) -> str:
    """
    클러스터의 문장들을 Coherence Evaluator에 전달하여 Good/Bad 판정을 받는다.
    """
    sampled = sample_from_cluster(cluster_sentences, n=CONFIG['cluster_sample_n'])
    prompt = f'입력: {json.dumps(sampled, ensure_ascii=False)}\n출력:'
    response = call_gemini(prompt, COHERENCE_SYSTEM)

    # 응답 정규화
    response_clean = response.strip().lower()
    if 'good' in response_clean:
        return 'Good'
    elif 'bad' in response_clean:
        return 'Bad'
    else:
        return 'Bad'  # 불확실한 경우 보수적으로 Bad 처리


print('Coherence Evaluator 정의 완료')

Coherence Evaluator 정의 완료


### 3-2. 단일 반복 실행 함수
Algorithm 1의 한 번의 iteration을 수행하는 함수.
후보 K 각각으로 클러스터링 → Coherence 평가 → 최적 K 선택 → Good/Bad 분리

In [ ]:
# === 단일 반복 실행 ===

def run_single_iteration(
    active_indices: np.ndarray,
    all_embeddings: np.ndarray,
    all_sentences: list,
    iteration: int,
) -> dict:
    """
    Algorithm 1의 단일 iteration을 수행한다.

    Parameters:
        active_indices: 현재 미할당 문장의 인덱스 배열
        all_embeddings: 전체 임베딩 행렬
        all_sentences: 전체 문장 리스트
        iteration: 현재 반복 번호 (0-indexed)

    Returns:
        dict with keys:
          'good_indices': 확정된 Good 클러스터 소속 인덱스 리스트의 리스트
          'remaining_indices': 다음 반복으로 넘어갈 인덱스 배열
          'best_k': 선택된 최적 K
          'log': 후보 K별 Good/Bad 통계 로그
    """
    active_emb = all_embeddings[active_indices]
    active_sents = [all_sentences[i] for i in active_indices]
    n_active = len(active_indices)

    print(f'\n{"="*60}')
    print(f'Iteration {iteration} | 활성 문장: {n_active:,}')
    print(f'{"="*60}')

    # 활성 문장 수보다 큰 K 후보는 제외
    candidate_k = [k for k in CONFIG['cluster_candidate_k'] if k < n_active]
    if not candidate_k:
        candidate_k = [max(2, n_active // 2)]
    print(f'후보 K: {candidate_k}')

    # 각 K에 대해 클러스터링 + Coherence 평가
    k_results = {}
    for k in candidate_k:
        print(f'\n--- K={k} ---')

        # K-Means 클러스터링
        kmeans = KMeans(
            n_clusters=k,
            n_init=CONFIG['cluster_kmeans_n_init'],
            random_state=CONFIG['cluster_random_state'] + iteration,
            max_iter=300,
        )
        labels = kmeans.fit_predict(active_emb)

        # 클러스터별 Coherence 평가
        good_count = 0
        bad_count = 0
        cluster_evals = []  # (cluster_id, 'Good'/'Bad', [member_indices])

        for cid in tqdm(range(k), desc=f'Coherence K={k}', leave=False):
            member_mask = labels == cid
            member_local_indices = np.where(member_mask)[0]
            member_sents = [active_sents[i] for i in member_local_indices]

            if len(member_sents) < 2:
                # 문장 1개 이하 클러스터는 Bad 처리
                verdict = 'Bad'
            else:
                verdict = evaluate_coherence(member_sents)

            if verdict == 'Good':
                good_count += 1
            else:
                bad_count += 1

            # active_indices 기준의 원본 인덱스로 변환
            member_global_indices = active_indices[member_local_indices].tolist()
            cluster_evals.append((cid, verdict, member_global_indices))

        ratio = good_count / (bad_count + 1)
        print(f'  Good: {good_count}, Bad: {bad_count}, Ratio: {ratio:.3f}')
        k_results[k] = {
            'good_count': good_count,
            'bad_count': bad_count,
            'ratio': ratio,
            'cluster_evals': cluster_evals,
        }

    # 최적 K 선택: Good/Bad 비율 최대화
    best_k = max(k_results, key=lambda k: k_results[k]['ratio'])
    best = k_results[best_k]
    print(f'\n★ 최적 K={best_k} 선택 (Good={best["good_count"]}, Bad={best["bad_count"]}, Ratio={best["ratio"]:.3f})')

    # Good 클러스터 확정, Bad 클러스터의 문장은 remaining
    good_clusters = []
    remaining_set = set()

    for cid, verdict, member_indices in best['cluster_evals']:
        if verdict == 'Good':
            good_clusters.append(member_indices)
        else:
            remaining_set.update(member_indices)

    remaining_indices = np.array(sorted(remaining_set))

    good_sentence_count = sum(len(c) for c in good_clusters)
    print(f'  Good 클러스터: {len(good_clusters)}개 ({good_sentence_count:,}문장)')
    print(f'  남은 문장: {len(remaining_indices):,} ({len(remaining_indices)/len(all_sentences)*100:.1f}%)')

    # 로그
    log = {str(k): {'good': v['good_count'], 'bad': v['bad_count'], 'ratio': round(v['ratio'], 3)}
           for k, v in k_results.items()}

    return {
        'good_clusters': good_clusters,
        'remaining_indices': remaining_indices,
        'best_k': best_k,
        'log': log,
    }


print('단일 반복 함수 정의 완료')

단일 반복 함수 정의 완료


### 3-3. 전체 반복 실행
Algorithm 1의 전체 루프를 실행한다. 반복마다 체크포인트를 저장하여 세션 끊김에 대비한다.

**체크포인트**: `checkpoints/clustering/iter_{t}.json` (반복별 Good 클러스터 + 남은 인덱스)

In [ ]:
# === 전체 반복 실행 ===

def load_iteration_checkpoint() -> tuple:
    """
    가장 최근 반복 체크포인트를 로드하여 (all_good_clusters, remaining_indices, start_iter)를 반환한다.
    체크포인트가 없으면 초기 상태를 반환한다.
    """
    ckpt_dir = PATH_CONFIG['clustering_ckpt_dir']
    ckpt_files = sorted(ckpt_dir.glob('iter_*.json'))

    if not ckpt_files:
        return [], np.arange(len(sentences)), 0

    # 가장 마지막 체크포인트
    latest = ckpt_files[-1]
    with open(latest, 'r') as f:
        data = json.load(f)

    all_good = [list(c) for c in data['all_good_clusters']]
    remaining = np.array(data['remaining_indices'])
    start_iter = data['completed_iteration'] + 1

    print(f'체크포인트 로드: {latest.name}')
    print(f'  완료 반복: {data["completed_iteration"]}, 확정 클러스터: {len(all_good)}, 남은 문장: {len(remaining):,}')

    return all_good, remaining, start_iter


def save_iteration_checkpoint(
    iteration: int,
    all_good_clusters: list,
    remaining_indices: np.ndarray,
    iter_log: dict,
) -> None:
    ckpt_path = PATH_CONFIG['clustering_ckpt_dir'] / f'iter_{iteration:02d}.json'
    data = {
        'completed_iteration': iteration,
        'n_good_clusters': len(all_good_clusters),
        'n_good_sentences': sum(len(c) for c in all_good_clusters),
        'n_remaining': len(remaining_indices),
        'all_good_clusters': [list(c) for c in all_good_clusters],
        'remaining_indices': remaining_indices.tolist(),
        'iter_log': iter_log,
    }
    with open(ckpt_path, 'w') as f:
        json.dump(data, f)
    print(f'체크포인트 저장: {ckpt_path.name}')


def run_iterative_clustering(
    all_embeddings: np.ndarray,
    all_sentences: list,
) -> list:
    """
    Algorithm 1 전체 루프를 실행한다.
    반환: 확정된 Good 클러스터 리스트 (각 클러스터는 인덱스 리스트)
    """
    all_good_clusters, remaining_indices, start_iter = load_iteration_checkpoint()
    total_n = len(all_sentences)

    for t in range(start_iter, CONFIG['cluster_max_iter']):
        # 종료 조건: 미할당 비율
        remaining_ratio = len(remaining_indices) / total_n
        if remaining_ratio < CONFIG['cluster_epsilon']:
            print(f'\n종료: 미할당 비율 {remaining_ratio:.3f} < ε={CONFIG["cluster_epsilon"]}')
            break

        if len(remaining_indices) < 10:
            print(f'\n종료: 남은 문장 {len(remaining_indices)}개 (최소 10개 필요)')
            break

        # 단일 반복 실행
        result = run_single_iteration(
            active_indices=remaining_indices,
            all_embeddings=all_embeddings,
            all_sentences=all_sentences,
            iteration=t,
        )

        all_good_clusters.extend(result['good_clusters'])
        remaining_indices = result['remaining_indices']

        # 체크포인트 저장
        save_iteration_checkpoint(t, all_good_clusters, remaining_indices, result['log'])

    # 최종 통계
    total_good_sents = sum(len(c) for c in all_good_clusters)
    print(f'\n{"="*60}')
    print(f'반복 클러스터링 완료')
    print(f'  확정 클러스터: {len(all_good_clusters)}')
    print(f'  확정 문장: {total_good_sents:,} / {total_n:,} ({total_good_sents/total_n*100:.1f}%)')
    print(f'  미할당 문장: {len(remaining_indices):,} ({len(remaining_indices)/total_n*100:.1f}%)')
    print(f'{"="*60}')

    return all_good_clusters


# 실행
good_clusters = run_iterative_clustering(embeddings, sentences)


Iteration 0 | 활성 문장: 34,025
후보 K: [50, 100, 200, 500, 800, 1000]

--- K=50 ---


Coherence K=50:   0%|          | 0/50 [00:00<?, ?it/s]

  Good: 31, Bad: 19, Ratio: 1.550

--- K=100 ---


Coherence K=100:   0%|          | 0/100 [00:00<?, ?it/s]

  Good: 79, Bad: 21, Ratio: 3.591

--- K=200 ---


...

  Good: 305, Bad: 195, Ratio: 1.556

--- K=800 ---


Coherence K=800:   0%|          | 0/800 [00:00<?, ?it/s]

  Good: 396, Bad: 404, Ratio: 0.978

--- K=1000 ---


Coherence K=1000:   0%|          | 0/1000 [00:00<?, ?it/s]

  Good: 417, Bad: 583, Ratio: 0.714

★ 최적 K=500 선택 (Good=305, Bad=195, Ratio=1.556)
  Good 클러스터: 305개 (1,629문장)
  남은 문장: 684 (2.0%)
체크포인트 저장: iter_01.json

종료: 미할당 비율 0.020 < ε=0.05

반복 클러스터링 완료
  확정 클러스터: 1030
  확정 문장: 33,341 / 34,025 (98.0%)
  미할당 문장: 684 (2.0%)


## 4. 의도 라벨링 (Cluster Naming)
확정된 Good 클러스터 각각에 "행위-목적" 형식의 의도 라벨을 생성한다.
논문 Section 3.1의 Cluster Naming 유틸리티를 Gemini API로 구현.

**체크포인트**: `checkpoints/clustering/labels.json`

In [ ]:
# === 의도 라벨링 ===

LABELLING_SYSTEM = """당신은 고객 상담 의도를 분류하는 전문가입니다.

주어진 문장 그룹의 공통 의도를 "행위-목적" 형식의 간결한 라벨로 요약하세요.

## 형식 규칙
- 반드시 "행위-목적" 형식 (하이픈으로 구분)
- 행위 예시: 문의, 확인요청, 요청, 변경요청, 해지요청, 신고, 신청, 결제요청
- 목적은 구체적으로 (예: "환불", "요금제변경", "배송상태", "카드분실")

## 예시
입력: ["문의 — 요금제 변경 방법", "문의 — 현재 요금제에서 다른 요금제로 변경", "문의 — 요금제 변경 가능 여부"]
출력: 문의-요금제변경

입력: ["확인 요청 — 카드 환불 누락", "확인 요청 — 환불 미처리 상태", "문의 — 환불 금액 미입금"]
출력: 확인요청-환불처리상태

입력: ["요청 — 선결제 즉시 출금", "결제 요청 — 이번 달 카드 대금 즉시 결제", "요청 — 결제 대금 선납"]
출력: 요청-선결제즉시출금

라벨만 출력하세요. 다른 내용은 포함하지 마세요."""


def label_clusters(good_clusters: list, all_sentences: list) -> list:
    """
    확정된 Good 클러스터에 "행위-목적" 라벨을 부여한다.
    체크포인트가 존재하면 로드.
    반환: [(cluster_indices, label), ...]
    """
    ckpt_path = PATH_CONFIG['clustering_ckpt_dir'] / 'labels.json'

    if ckpt_path.exists():
        print(f'라벨 체크포인트 로드: {ckpt_path}')
        with open(ckpt_path, 'r') as f:
            data = json.load(f)
        return [(d['indices'], d['label']) for d in data]

    labeled = []
    for i, cluster_indices in enumerate(tqdm(good_clusters, desc='라벨링')):
        cluster_sents = [all_sentences[idx] for idx in cluster_indices]
        sampled = sample_from_cluster(cluster_sents, n=CONFIG['cluster_sample_n'])
        prompt = f'입력: {json.dumps(sampled, ensure_ascii=False)}\n출력:'
        label = call_gemini(prompt, LABELLING_SYSTEM, max_output_tokens=CONFIG['label_max_output_tokens'])

        if not label:
            label = f'미분류-클러스터{i}'

        label = nfc(label.strip().strip('"').strip("'"))
        labeled.append((cluster_indices, label))

    # 체크포인트 저장
    save_data = [{'indices': list(indices), 'label': label} for indices, label in labeled]
    with open(ckpt_path, 'w', encoding='utf-8') as f:
        json.dump(save_data, f, ensure_ascii=False, indent=2)
    print(f'라벨 체크포인트 저장: {ckpt_path}')
    print(f'총 {len(labeled)}개 클러스터 라벨링 완료')

    return labeled


labeled_clusters = label_clusters(good_clusters, sentences)

# 라벨 분포 확인
labels_only = [lbl for _, lbl in labeled_clusters]
print(f'\n고유 라벨 수: {len(set(labels_only))}')
print(f'상위 10개 라벨:')
from collections import Counter
for lbl, cnt in Counter(labels_only).most_common(10):
    print(f'  {lbl}: {cnt}')

라벨링:   0%|          | 0/1030 [00:00<?, ?it/s]

  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp... (1/3)
  [Error] 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp... (1/3)
라벨 체크포인트 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/clustering/labels.json
총 1030개 클러스터 라벨링 완료

고유 라벨 수: 926
상위 10개 라벨:
  문의-요금제변경: 6
  문의-요금제: 5
  문의-결제방법: 5
  요청-선결제: 4
  문의-공항픽업서비스: 3
  문의-현금서비스: 3
  문의-연회비: 3
  확인요청-미납요금: 3
  확인요청-자동이체: 3
  요청-가상계좌발급: 3


## 5. 클러스터 병합 (Post-Correction)
논문 Section 3.3의 post-correction을 구현한다.
라벨 임베딩 간 geodesic distance가 θ 미만이고, von Mises-Fisher 기반 병합 확률이 τ 이상인 클러스터 쌍을 자동 병합한다.

**체크포인트**: `checkpoints/clustering/merged_clusters.json`

In [ ]:
# === 클러스터 병합 ===
from sentence_transformers import SentenceTransformer
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix


def compute_geodesic_distance(emb_a: np.ndarray, emb_b: np.ndarray) -> float:
    """단위 초구면 위 두 벡터 간 geodesic distance (각도 거리)."""
    cos_sim = np.clip(np.dot(emb_a, emb_b), -1.0, 1.0)
    return np.arccos(cos_sim)


def vmf_log_prob(emb: np.ndarray, mu: np.ndarray, kappa: float, d: int) -> float:
    """von Mises-Fisher 분포의 로그 확률 밀도."""
    log_norm = (d / 2 - 1) * np.log(kappa) - (d / 2) * np.log(2 * np.pi) - np.log(bessel_iv(d / 2 - 1, kappa) + 1e-300)
    return log_norm + kappa * np.dot(emb, mu)


def compute_merge_probability(
    emb_i: np.ndarray,
    emb_j: np.ndarray,
    all_label_embs: np.ndarray,
    kappa: float,
) -> float:
    """
    논문 Eq. P(same | l_i, l_j): 두 라벨이 같은 의도에서 왔을 확률.
    모든 라벨 임베딩을 mixture center로 사용.
    """
    K = len(all_label_embs)
    d = emb_i.shape[0]
    pi_m = 1.0 / K  # uniform weights

    total_prob = 0.0
    for mu_m in all_label_embs:
        log_p_i = vmf_log_prob(emb_i, mu_m, kappa, d)
        log_p_j = vmf_log_prob(emb_j, mu_m, kappa, d)
        total_prob += pi_m * np.exp(log_p_i + log_p_j)

    return total_prob


def merge_clusters(
    labeled_clusters: list,
    all_sentences: list,
) -> list:
    """
    라벨 임베딩 기반으로 유사 클러스터를 병합한다.
    체크포인트가 존재하면 로드.

    반환: [(merged_indices, merged_label), ...]
    """
    ckpt_path = PATH_CONFIG['clustering_ckpt_dir'] / 'merged_clusters.json'

    if ckpt_path.exists():
        print(f'병합 체크포인트 로드: {ckpt_path}')
        with open(ckpt_path, 'r') as f:
            data = json.load(f)
        return [(d['indices'], d['label']) for d in data]

    # 라벨 임베딩 생성
    labels = [lbl for _, lbl in labeled_clusters]
    print(f'라벨 임베딩 생성 중... ({len(labels)}개)')
    embed_model = SentenceTransformer(CONFIG['embed_model_name'])
    label_embs = embed_model.encode(labels, normalize_embeddings=True)

    K = len(labeled_clusters)
    d = label_embs.shape[1]
    theta = CONFIG['merge_geodesic_theta']
    tau = CONFIG['merge_prob_tau']
    kappa = CONFIG['merge_kappa']

    # Affinity graph 구축
    print(f'Affinity graph 구축... (θ={theta}, τ={tau})')
    edges_i, edges_j = [], []

    for i in tqdm(range(K), desc='병합 후보 탐색'):
        for j in range(i + 1, K):
            dist = compute_geodesic_distance(label_embs[i], label_embs[j])
            if dist < theta:
                prob = compute_merge_probability(label_embs[i], label_embs[j], label_embs, kappa)
                if prob > tau:
                    edges_i.append(i)
                    edges_j.append(j)

    print(f'병합 엣지: {len(edges_i)}개')

    # Connected components로 병합
    if edges_i:
        data_ones = np.ones(len(edges_i) * 2)
        rows = edges_i + edges_j
        cols = edges_j + edges_i
        graph = csr_matrix((data_ones, (rows, cols)), shape=(K, K))
        n_components, component_labels = connected_components(graph, directed=False)
    else:
        n_components = K
        component_labels = np.arange(K)

    # 병합 실행
    merged = []
    for comp_id in range(n_components):
        member_cluster_ids = np.where(component_labels == comp_id)[0]

        # 인덱스 합치기
        merged_indices = []
        for cid in member_cluster_ids:
            merged_indices.extend(labeled_clusters[cid][0])

        # 라벨: 가장 큰 클러스터의 라벨 사용
        biggest_cid = max(member_cluster_ids, key=lambda c: len(labeled_clusters[c][0]))
        merged_label = labeled_clusters[biggest_cid][1]

        merged.append((merged_indices, merged_label))

    print(f'\n병합 결과: {K} → {len(merged)} 클러스터 ({K - len(merged)}개 병합됨)')

    # 체크포인트 저장
    save_data = [{'indices': list(indices), 'label': label} for indices, label in merged]
    with open(ckpt_path, 'w', encoding='utf-8') as f:
        json.dump(save_data, f, ensure_ascii=False, indent=2)
    print(f'체크포인트 저장: {ckpt_path}')

    return merged


merged_clusters = merge_clusters(labeled_clusters, sentences)

라벨 임베딩 생성 중... (1030개)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: snunlp/KR-SBERT-V40K-klueNLI-augSTS
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Affinity graph 구축... (θ=0.8, τ=0.7)


병합 후보 탐색:   0%|          | 0/1030 [00:00<?, ?it/s]

병합 엣지: 15437개

병합 결과: 1030 → 99 클러스터 (931개 병합됨)
체크포인트 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/checkpoints/clustering/merged_clusters.json


## 6. 결과 검증 및 통계
최종 IntentSet의 품질과 분포를 확인한다.

In [ ]:
# === 결과 검증 ===

def validate_intentset(
    merged_clusters: list,
    all_sentences: list,
    intent_df: pd.DataFrame,
) -> dict:
    """
    최종 IntentSet의 통계를 산출하고 원본 카테고리와의 일관성을 확인한다.
    """
    stats = {}

    # 기본 통계
    n_clusters = len(merged_clusters)
    total_assigned = sum(len(c[0]) for c in merged_clusters)
    total_sentences = len(all_sentences)

    print(f'=== 초기 IntentSet 통계 ===')
    print(f'  클러스터 수: {n_clusters}')
    print(f'  할당된 문장: {total_assigned:,} / {total_sentences:,} ({total_assigned/total_sentences*100:.1f}%)')
    stats['n_clusters'] = n_clusters
    stats['coverage'] = round(total_assigned / total_sentences * 100, 1)

    # 클러스터 크기 분포
    sizes = [len(c[0]) for c in merged_clusters]
    print(f'\n--- 클러스터 크기 분포 ---')
    print(f'  평균: {np.mean(sizes):.1f}, 중앙값: {np.median(sizes):.0f}')
    print(f'  최소: {np.min(sizes)}, 최대: {np.max(sizes)}')
    print(f'  1~5문장 클러스터: {sum(1 for s in sizes if s <= 5)}')
    print(f'  6~20문장 클러스터: {sum(1 for s in sizes if 6 <= s <= 20)}')
    print(f'  21~100문장 클러스터: {sum(1 for s in sizes if 21 <= s <= 100)}')
    print(f'  100+문장 클러스터: {sum(1 for s in sizes if s > 100)}')

    # 고유 라벨 수
    unique_labels = set(lbl for _, lbl in merged_clusters)
    print(f'\n  고유 라벨: {len(unique_labels)} (클러스터 {n_clusters}개 중)')

    # 행위 접두사 분포
    print(f'\n--- 행위 접두사 분포 ---')
    action_counts = Counter()
    for _, lbl in merged_clusters:
        action = lbl.split('-')[0] if '-' in lbl else lbl
        action_counts[action] += 1
    for action, cnt in action_counts.most_common(15):
        print(f'  {action}: {cnt}')

    # 상위 20개 라벨 (크기순)
    print(f'\n--- 상위 20 클러스터 (크기순) ---')
    sorted_clusters = sorted(merged_clusters, key=lambda c: len(c[0]), reverse=True)
    for indices, label in sorted_clusters[:20]:
        print(f'  [{len(indices):>4}문장] {label}')

    # 원본 consulting_category와 교차 확인 (샘플)
    print(f'\n--- 원본 카테고리 교차 확인 (상위 5 클러스터) ---')
    for indices, label in sorted_clusters[:5]:
        original_cats = intent_df.iloc[indices]['consulting_category'].value_counts().head(3)
        print(f'  [{label}] → {dict(original_cats)}')

    return stats


intentset_stats = validate_intentset(merged_clusters, sentences, intent_df)

=== 초기 IntentSet 통계 ===
  클러스터 수: 99
  할당된 문장: 33,341 / 34,025 (98.0%)

--- 클러스터 크기 분포 ---
  평균: 336.8, 중앙값: 8
  최소: 2, 최대: 31596
  1~5문장 클러스터: 44
  6~20문장 클러스터: 25
  21~100문장 클러스터: 27
  100+문장 클러스터: 3

  고유 라벨: 99 (클러스터 99개 중)

--- 행위 접두사 분포 ---
  문의: 68
  요청: 12
  확인요청: 11
  변경요청: 2
  정보제공: 2
  동의: 1
  신청: 1
  거절: 1
  정보변경: 1

--- 상위 20 클러스터 (크기순) ---
  [31596문장] 문의-요금제변경가능여부
  [ 153문장] 동의-신용정보조회
  [ 112문장] 문의-풀빌라예약정보
  [  86문장] 변경요청-자동이체계좌
  [  61문장] 확인요청-자녀명의요금및결제
  [  56문장] 문의-조식포함여부
  [  50문장] 확인요청-아파트관리비
  [  50문장] 문의-물리아리조트패키지
  [  50문장] 확인요청-LGU+관련정보
  [  43문장] 요청-부서연결
  [  41문장] 문의-레이트체크아웃
  [  40문장] 문의-픽업서비스
  [  40문장] 문의-명세서
  [  40문장] 문의-보이스피싱대처
  [  39문장] 확인요청-외국인등록증제출
  [  37문장] 문의-실적제외항목
  [  36문장] 문의-잔액부족
  [  34문장] 문의-영화월정액
  [  33문장] 문의-캠핀스키호텔숙박예약
  [  33문장] 문의-아동관련

--- 원본 카테고리 교차 확인 (상위 5 클러스터) ---
  [문의-요금제변경가능여부] → {'선결제/즉시출금': np.int64(2890), '요금 안내': np.int64(2246), '이용내역 안내 ': np.int64(2106)}
  [동의-신용정보조회] → {'한도상향 접수/처리 ': np.int64(146), '특별한도/임시한도/일시한도 안내/신청

## 7. 최종 IntentSet 저장
각 의도 레코드에 클러스터 ID와 라벨을 부여하여 `data/processed/initial_intentset.parquet`에 저장한다.

In [ ]:
# === 최종 IntentSet 저장 ===

def save_initial_intentset(
    merged_clusters: list,
    intent_df: pd.DataFrame,
) -> pd.DataFrame:
    """
    최종 IntentSet을 DataFrame으로 구성하여 저장한다.
    """
    # 인덱스 → 클러스터 ID + 라벨 매핑
    idx_to_cluster = {}
    idx_to_label = {}

    for cluster_id, (indices, label) in enumerate(merged_clusters):
        for idx in indices:
            idx_to_cluster[idx] = cluster_id
            idx_to_label[idx] = label

    # intent_df에 클러스터 정보 추가
    result_df = intent_df.copy()
    result_df['cluster_id'] = result_df.index.map(lambda i: idx_to_cluster.get(i, -1))
    result_df['intent_label'] = result_df.index.map(lambda i: idx_to_label.get(i, 'UNASSIGNED'))

    # 통계
    assigned = result_df[result_df['cluster_id'] >= 0]
    unassigned = result_df[result_df['cluster_id'] < 0]
    print(f'할당됨: {len(assigned):,}, 미할당: {len(unassigned):,}')

    # 저장
    output_path = PATH_CONFIG['initial_intentset']
    result_df.to_parquet(output_path, index=False)
    print(f'저장: {output_path}')

    # 클러스터 요약 테이블도 별도 저장
    cluster_summary = []
    for cluster_id, (indices, label) in enumerate(merged_clusters):
        cluster_summary.append({
            'cluster_id': cluster_id,
            'intent_label': label,
            'size': len(indices),
            'sample_sentences': json.dumps(
                [intent_df.iloc[i]['intent_summary'] for i in indices[:5]],
                ensure_ascii=False,
            ),
        })

    summary_df = pd.DataFrame(cluster_summary)
    summary_path = PATH_CONFIG['processed'] / 'intentset_summary.parquet'
    summary_df.to_parquet(summary_path, index=False)
    print(f'클러스터 요약 저장: {summary_path}')

    return result_df


final_df = save_initial_intentset(merged_clusters, intent_df)
final_df.head()

할당됨: 33,341, 미할당: 684
저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/initial_intentset.parquet
클러스터 요약 저장: /content/drive/MyDrive/5stone_experiment/1_clustering_test/data/processed/intentset_summary.parquet


,source_id,source,consulting_category,split,intent_index,intent_summary,relevant_utterances,n_intents,is_fallback,raw_response,cluster_id,intent_label
0,90100,액티벤처,상품예약 및 결제,train,0,문의 — ▲▲월 ▲▲일 출발 5일 일정 2명 예약 가능 여부,"[""안녕하세요, ▲▲월 ▲▲일 출발로 5일 일정으로 2명 예약 가능한지 궁금해요.""...",8,False,,0,문의-요금제변경가능여부
1,90100,액티벤처,상품예약 및 결제,train,1,문의 — 비치 좋은 리조트 조건 및 제외 지역,"[""비치가 좋은 곳으로, 쿠타 쪽으로 가는 택시가 힘들지 않았으면 좋겠어요."", ""...",8,False,,0,문의-요금제변경가능여부
2,90100,액티벤처,상품예약 및 결제,train,2,문의 — 희망 호텔 및 숙박 형태,"[""호텔은 그랜드 하얏트나 미라지를 생각하고 있어요."", ""자매 두 명이라 풀빌라를...",8,False,,0,문의-요금제변경가능여부
3,90100,액티벤처,상품예약 및 결제,train,3,요청 — 추천 호텔 문의,"[""추천해주실 만한 곳이 있으면 알려주세요.""]",8,False,,0,문의-요금제변경가능여부
4,90100,액티벤처,상품예약 및 결제,train,4,요청 — 담당자 직통 전화번호 문의,"[""담당자 직통 전화번호도 부탁드려요."", ""직통전화번호도 다시 한 번 알려주실 수...",8,False,,0,문의-요금제변경가능여부
